In [1]:
%mamba install scikit-learn matplotlib seaborn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, scikit-learn, seaborn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.421 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ ca-certificates     2026.10.23  hbd8a1cb_0           conda-forge
+ certifi             2026.10.23  pyhd8ed1ab_3         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.28.0      pyh5ded981_0         conda-forge
+ pandas              3.1.0       np23py313h124dbc7_0  emscripten-forge-4x
+ patsy               1.0.3       py313h1804a44_0      emscripten-forge-4x
+ requests            2.34.2      pyhcf101f3_0         conda-forge
+ scikit-learn  

In [2]:
import json
import random
import urllib.request
import numpy as np
import pandas as pd

np.random.seed(42)
random.seed(42)

# ==============================================================================
# PART 1: SIMULATE BASE FINANCIAL DATA (1,200 RECORDS)
# ==============================================================================
NUM_RECORDS = 1200
data = []

for i in range(NUM_RECORDS):
    customer_id = f"CUST_{10000 + i}"
    age = int(np.clip(np.random.normal(38, 11), 21, 70))
    annual_income = round(np.clip(np.random.lognormal(11.0, 0.55), 24000, 250000), 2)
    credit_score = int(np.clip(np.random.normal(660, 85), 350, 850))
    loan_amount = round(np.clip(annual_income * np.random.uniform(0.15, 0.85), 5000, 150000), 2)
    debt_to_income = round(np.clip(np.random.beta(2, 5) * 0.85, 0.05, 0.80), 3)
    late_payments_12m = int(np.random.poisson(0.6 if credit_score > 650 else 2.2))
    account_tenure_months = int(np.random.randint(3, 120))
    employment_status = np.random.choice(
        ["Full-Time", "Self-Employed", "Part-Time", "Contract"],
        p=[0.65, 0.18, 0.10, 0.07]
    )
    
    risk_score = (
        (700 - credit_score) * 0.008 +
        (debt_to_income * 4.5) +
        (late_payments_12m * 0.65) +
        (0.5 if employment_status in ["Part-Time", "Contract"] else 0.0) -
        (account_tenure_months * 0.01) +
        np.random.normal(0, 0.6)
    )
    default_prob = 1 / (1 + np.exp(-risk_score))
    loan_default = 1 if default_prob > 0.52 else 0

    data.append({
        "customer_id": customer_id,
        "age": age,
        "annual_income": annual_income,
        "credit_score": credit_score,
        "loan_amount": loan_amount,
        "debt_to_income_ratio": debt_to_income,
        "late_payments_12m": late_payments_12m,
        "account_tenure_months": account_tenure_months,
        "employment_status": employment_status,
        "loan_default": loan_default
    })

df = pd.DataFrame(data)

# ==============================================================================
# PART 2: GEN-AI TEXTUAL DATA GENERATION
# ==============================================================================
high_risk_remarks = [
    "Customer requested a payment extension due to unexpected medical bills and reduced work hours.",
    "Multiple follow-ups needed for overdue installments; applicant expressed stress over rising monthly expenses.",
    "Applicant is consolidating high-interest credit card debt and recently changed jobs.",
    "Customer complained about late fee charges while citing temporary cash flow shortages.",
    "Inconsistent income deposits observed over the past quarter; borrower asked for restructuring options.",
    "Client stated business revenue dropped significantly this season, making full repayments difficult."
]
moderate_remarks = [
    "Loan requested for home renovation; steady income but existing auto loan increases debt burden.",
    "Customer inquired about flexible repayment dates to better align with bi-weekly payroll.",
    "Applicant has a solid employment history, though credit utilization has been climbing recently.",
    "Requested funds to expand small side business; provided partial tax documentation."
]
low_risk_remarks = [
    "Long-standing customer applying for equipment upgrade; maintains strong savings balance and auto-debit.",
    "Applicant consistently pays ahead of schedule and requested lower interest rate tier.",
    "Stable salaried professional seeking personal loan for family travel; zero history of delinquency.",
    "Very responsive during verification; strong debt coverage ratio and clean repayment track record.",
    "Customer praised mobile banking support and set up automatic monthly transfers immediately."
]

def assign_remark(row):
    if row["loan_default"] == 1 or row["late_payments_12m"] >= 2:
        return random.choice(high_risk_remarks)
    elif row["debt_to_income_ratio"] > 0.40:
        return random.choice(moderate_remarks)
    else:
        return random.choice(low_risk_remarks)

df["customer_remark"] = df.apply(assign_remark, axis=1)

# ==============================================================================
# PART 3: LLM / SLM FEATURE EXTRACTION (GROQ API OR BUILT-IN FALLBACK)
# ==============================================================================
GROQ_API_KEY = ""  # Optional: Paste a free Groq API key (gsk_...) to run Llama-3.1-8B SLM

def extract_features_via_slm(unique_texts):
    mapping = {}
    if GROQ_API_KEY:
        url = "https://api.groq.com/openai/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        }
        for text in unique_texts:
            prompt = (
                f"Analyze this bank customer remark and return ONLY valid JSON with keys "
                f"'sentiment' (POSITIVE or NEGATIVE), 'sentiment_score' (-1.0 to 1.0), "
                f"'risk_category' (High Financial Distress, Moderate Debt Burden, or Financially Stable), "
                f"and 'customer_segment' (Debt Consolidator, Growth & Investment, or Routine Borrower).\n"
                f"Remark: \"{text}\""
            )
            payload = json.dumps({
                "model": "llama-3.1-8b-instant",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.1,
                "response_format": {"type": "json_object"}
            }).encode("utf-8")
            req = urllib.request.Request(url, data=payload, headers=headers, method="POST")
            with urllib.request.urlopen(req) as resp:
                res = json.loads(resp.read().decode("utf-8"))
                mapping[text] = json.loads(res["choices"][0]["message"]["content"])
    else:
        for text in unique_texts:
            if any(w in text.lower() for w in ["overdue", "shortages", "dropped", "extension", "stress", "consolidating"]):
                mapping[text] = {
                    "sentiment": "NEGATIVE",
                    "sentiment_score": round(random.uniform(-0.95, -0.65), 3),
                    "risk_category": "High Financial Distress",
                    "customer_segment": "Debt Consolidator"
                }
            elif any(w in text.lower() for w in ["burden", "climbing", "partial", "expand"]):
                mapping[text] = {
                    "sentiment": "NEGATIVE",
                    "sentiment_score": round(random.uniform(-0.45, -0.10), 3),
                    "risk_category": "Moderate Debt Burden",
                    "customer_segment": "Growth & Investment"
                }
            else:
                mapping[text] = {
                    "sentiment": "POSITIVE",
                    "sentiment_score": round(random.uniform(0.75, 0.98), 3),
                    "risk_category": "Financially Stable",
                    "customer_segment": "Routine Borrower"
                }
    return mapping

unique_remarks = df["customer_remark"].unique()
feature_map = extract_features_via_slm(unique_remarks)

df["ai_sentiment_label"] = df["customer_remark"].map(lambda x: feature_map[x]["sentiment"])
df["ai_sentiment_score"] = df["customer_remark"].map(lambda x: feature_map[x]["sentiment_score"])
df["ai_risk_category"] = df["customer_remark"].map(lambda x: feature_map[x]["risk_category"])
df["ai_customer_segment"] = df["customer_remark"].map(lambda x: feature_map[x]["customer_segment"])

# ==============================================================================
# PART 4: SAVE & DOWNLOAD
# ==============================================================================
df.to_csv("financial_loan_ai_dataset.csv", index=False)
print("Done! Generated financial_loan_ai_dataset.csv with shape:", df.shape)
df.head()

Done! Generated financial_loan_ai_dataset.csv with shape: (1200, 15)


,customer_id,age,annual_income,credit_score,loan_amount,debt_to_income_ratio,late_payments_12m,account_tenure_months,employment_status,loan_default,customer_remark,ai_sentiment_label,ai_sentiment_score,ai_risk_category,ai_customer_segment
0,CUST_10000,43,55489.81,715,14383.68,0.284,2,66,Contract,1,Client stated business revenue dropped signifi...,NEGATIVE,-0.661,High Financial Distress,Debt Consolidator
1,CUST_10001,54,139566.05,452,25493.97,0.326,4,75,Full-Time,1,Customer requested a payment extension due to ...,NEGATIVE,-0.912,High Financial Distress,Debt Consolidator
2,CUST_10002,31,100819.99,684,46186.32,0.181,0,106,Self-Employed,0,Long-standing customer applying for equipment ...,POSITIVE,0.972,Financially Stable,Routine Borrower
3,CUST_10003,22,49305.25,620,36466.13,0.184,2,64,Full-Time,1,Client stated business revenue dropped signifi...,NEGATIVE,-0.661,High Financial Distress,Debt Consolidator
4,CUST_10004,21,40299.15,620,13969.87,0.331,3,13,Full-Time,1,Applicant is consolidating high-interest credi...,NEGATIVE,-0.850,High Financial Distress,Debt Consolidator


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

def z_score(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / np.std(x)

def draw_in_range(rng, draw_fn, low, high, is_int=False):
    val = draw_fn()
    while val < low or val > high:
        val = draw_fn()
    return int(round(val)) if is_int else val

def main():
    rng = np.random.default_rng(42)
    n = 1200

    # STEP 1 - CUSTOMER FEATURES
    customer_id = [f"CUST_{10000 + i}" for i in range(n)]

    # age: integer, normal(mean 38, sd 10.5), range 21-70
    age = np.array([
        draw_in_range(rng, lambda: rng.normal(38, 10.5), 21, 70, is_int=True)
        for _ in range(n)
    ])

    # employment_status: Full-Time 60%, Self-Employed 18%, Part-Time 12%, Contract 10%
    emp_choices = ["Full-Time", "Self-Employed", "Part-Time", "Contract"]
    emp_probs = [0.60, 0.18, 0.12, 0.10]
    employment_status = rng.choice(emp_choices, size=n, p=emp_probs)

    # annual_income
    emp_medians = {
        "Full-Time": 62000.0,
        "Self-Employed": 66000.0,
        "Part-Time": 34000.0,
        "Contract": 48000.0
    }
    annual_income = np.zeros(n)
    for i in range(n):
        med = emp_medians[employment_status[i]]
        age_factor = 1.0 + 0.004 * (age[i] - 38)
        val = draw_in_range(
            rng,
            lambda: med * np.exp(rng.normal(0, 0.45)) * age_factor,
            18000.0,
            250000.0,
            is_int=False
        )
        annual_income[i] = round(val, 2)

    # credit_score: integer, 660 + 20 * z(log income) + 10 * z(age) + normal(0, 75), range 300-850
    z_log_inc = z_score(np.log(annual_income))
    z_age = z_score(age)
    credit_score = np.zeros(n, dtype=int)
    for i in range(n):
        base = 660 + 20 * z_log_inc[i] + 10 * z_age[i]
        credit_score[i] = draw_in_range(
            rng,
            lambda: base + rng.normal(0, 75),
            300,
            850,
            is_int=True
        )

    # late_payments_12m: Poisson with rate exp(0.1 - 0.6 * z(credit_score)), max 8
    z_cs = z_score(credit_score)
    late_payments_12m = np.zeros(n, dtype=int)
    for i in range(n):
        rate = np.exp(0.1 - 0.6 * z_cs[i])
        late_payments_12m[i] = draw_in_range(
            rng,
            lambda: rng.poisson(rate),
            0,
            8,
            is_int=True
        )

    # account_tenure_months: random integer from 3 to min(120, (age - 18) * 12)
    account_tenure_months = np.array([
        int(rng.integers(3, min(120, (age[i] - 18) * 12) + 1))
        for i in range(n)
    ])

    # loan_amount: annual_income * exp(normal(-0.9, 0.6)), range 5000-150000, 2 decimals
    loan_amount = np.zeros(n)
    for i in range(n):
        val = draw_in_range(
            rng,
            lambda: annual_income[i] * np.exp(rng.normal(-0.9, 0.6)),
            5000.0,
            150000.0,
            is_int=False
        )
        loan_amount[i] = round(val, 2)

    # debt_to_income_ratio: (0.02 + 0.45 * Beta(2, 6)) + 0.25 * loan_amount / annual_income, range 0.03-0.80, 3 decimals
    debt_to_income_ratio = np.zeros(n)
    for i in range(n):
        ratio_term = 0.25 * loan_amount[i] / annual_income[i]
        val = draw_in_range(
            rng,
            lambda: (0.02 + 0.45 * rng.beta(2, 6)) + ratio_term,
            0.03,
            0.80,
            is_int=False
        )
        debt_to_income_ratio[i] = round(val, 3)

    # STEP 2 - HIDDEN VARIABLES
    stress = rng.normal(0, 1, size=n)
    purposes = [
        "Debt Consolidation",
        "Business or Investment",
        "Home Improvement",
        "Education",
        "Medical or Emergency",
        "Personal or Travel",
        "Vehicle"
    ]
    loan_purpose = rng.choice(purposes, size=n)

    # STEP 3 - TARGET (loan_default)
    z_lp = z_score(late_payments_12m)
    z_dti = z_score(debt_to_income_ratio)
    z_ten = z_score(account_tenure_months)

    emp_effect_map = {
        "Full-Time": 0.0,
        "Self-Employed": 0.15,
        "Part-Time": 0.25,
        "Contract": 0.30
    }
    emp_effect = np.array([emp_effect_map[e] for e in employment_status])

    purpose_effect = np.array([
        0.2 if p in ("Debt Consolidation", "Medical or Emergency") else 0.0
        for p in loan_purpose
    ])

    first_four = -0.9 * z_cs + 0.7 * z_lp + 0.6 * z_dti - 0.3 * z_ten
    risk = first_four + emp_effect + purpose_effect + 0.6 * stress

    # Solve for b0 so mean probability is 0.20
    b0 = brentq(lambda b: np.mean(1.0 / (1.0 + np.exp(-(b + risk)))) - 0.20, -10.0, 10.0)
    prob = 1.0 / (1.0 + np.exp(-(b0 + risk)))
    loan_default = rng.binomial(1, prob)

    # STEP 4 - CUSTOMER REMARK
    numeric_risk = z_score(first_four)
    tone = 0.6 * stress + 0.4 * numeric_risk + rng.normal(0, 0.6, size=n)
    p25, p50, p70, p88 = np.percentile(tone, [25, 50, 70, 88])
    tone_tier = np.zeros(n, dtype=int)
    for i in range(n):
        if tone[i] <= p25:
            tone_tier[i] = 1
        elif tone[i] <= p50:
            tone_tier[i] = 2
        elif tone[i] <= p70:
            tone_tier[i] = 3
        elif tone[i] <= p88:
            tone_tier[i] = 4
        else:
            tone_tier[i] = 5

    purpose_phrases = {
        "Debt Consolidation": [
            "Applicant seeks funds to combine several existing balances into one monthly payment",
            "Client is requesting a facility to streamline multiple credit obligations",
            "Loan requested to pay off scattered high-interest cards and simplify bills",
            "Borrower aims to merge outstanding personal debts under a single schedule",
            "Application is for consolidating various revolving credit lines",
            "Customer wants to restructure existing liabilities into one manageable loan",
            "Purpose stated is to clear multiple small balances with one term facility",
            "Client hopes to reduce administrative hassle by unifying current debts",
            "Applicant plans to settle several card accounts using this facility",
            "Request is intended to bundle existing consumer debts together",
            "Borrower is looking to replace multiple payments with one fixed installment",
            "Loan will be used to tidy up and combine current unsecured balances"
        ],
        "Business or Investment": [
            "Applicant is seeking capital to expand a small commercial venture",
            "Client intends to use proceeds for working capital and inventory",
            "Loan requested to fund equipment upgrades for an ongoing enterprise",
            "Borrower plans to invest in a new business opportunity",
            "Application supports expansion plans for the client's sole proprietorship",
            "Customer is looking for financing to scale up side-business operations",
            "Purpose stated is to purchase stock and cover initial project outlays",
            "Client wants to inject funds into a growing local service business",
            "Applicant seeks funding for commercial premises and operational setup",
            "Request is aimed at supporting a new investment initiative",
            "Borrower intends to allocate proceeds toward business development",
            "Loan will be applied to commercial growth and vendor payments"
        ],
        "Home Improvement": [
            "Applicant plans to renovate the kitchen and update living areas",
            "Client is requesting funds for residential repairs and roof maintenance",
            "Loan requested to finance home remodeling and energy-efficient upgrades",
            "Borrower intends to carry out essential plumbing and electrical work",
            "Application is for extending the family residence and adding a room",
            "Customer wants to modernize an older property they currently own",
            "Purpose stated is interior refurbishment and flooring replacement",
            "Client seeks financing for backyard landscaping and exterior painting",
            "Applicant is planning structural upgrades to their primary home",
            "Request is for funding necessary household repairs and fixtures",
            "Borrower hopes to complete a long-planned bathroom renovation",
            "Loan will be used to improve home insulation and window fittings"
        ],
        "Education": [
            "Applicant is seeking funds to cover upcoming university tuition fees",
            "Client intends to finance a postgraduate degree program",
            "Loan requested to pay for professional certification and coursework",
            "Borrower plans to support a dependent's college enrollment expenses",
            "Application is for funding specialized technical training",
            "Customer wants to cover academic fees and study materials",
            "Purpose stated is continuing education and skill development",
            "Client is looking to finance an executive MBA program",
            "Applicant seeks support for vocational school tuition",
            "Request is aimed at covering semester fees and campus housing",
            "Borrower intends to invest in formal academic qualifications",
            "Loan will be used for educational courses and exam registration"
        ],
        "Medical or Emergency": [
            "Applicant needs funds to cover unexpected medical treatment costs",
            "Client is requesting urgent financing for a family health procedure",
            "Loan requested to manage unforeseen hospital and clinic bills",
            "Borrower faces an immediate personal emergency requiring quick liquidity",
            "Application is for covering dental surgery and specialist care",
            "Customer wants to settle out-of-pocket healthcare expenses",
            "Purpose stated is addressing an urgent family medical situation",
            "Client seeks prompt assistance for unanticipated emergency outlays",
            "Applicant needs to bridge sudden healthcare-related costs",
            "Request is for urgent personal expenses following an unexpected event",
            "Borrower intends to pay for necessary diagnostic and treatment fees",
            "Loan will be applied to immediate medical and recovery bills"
        ],
        "Personal or Travel": [
            "Applicant is planning an overseas family trip and related expenses",
            "Client seeks funds for a major personal milestone celebration",
            "Loan requested to cover wedding arrangements and family travel",
            "Borrower intends to finance an upcoming international vacation",
            "Application is for personal discretionary spending and holiday plans",
            "Customer wants to fund a long-awaited sabbatical journey",
            "Purpose stated is covering travel tickets and accommodation",
            "Client is looking to finance a family reunion event",
            "Applicant plans to use proceeds for personal leisure and travel",
            "Request is for funding a special anniversary trip",
            "Borrower hopes to cover relocation and personal travel costs",
            "Loan will be used for personal plans and upcoming holiday bookings"
        ],
        "Vehicle": [
            "Applicant is seeking financing to purchase a pre-owned family car",
            "Client intends to buy a reliable commuter vehicle for daily work",
            "Loan requested to replace an aging car with a newer model",
            "Borrower plans to acquire a utility vehicle for household use",
            "Application is for funding a used sedan from a local dealer",
            "Customer wants to finance a motorcycle for daily transit",
            "Purpose stated is purchasing a dependable vehicle for family transport",
            "Client is looking to upgrade their current car",
            "Applicant seeks funds for a private vehicle purchase",
            "Request is aimed at acquiring a fuel-efficient hatchback",
            "Borrower intends to buy a second-hand SUV",
            "Loan will be used to complete the purchase of a personal vehicle"
        ]
    }

    tier_phrases = {
        1: [
            "Finances appear very well organized with comfortable monthly buffers",
            "Applicant demonstrates strong savings habits and steady cash flow",
            "Overall financial profile looks solid with minimal strain",
            "Client maintains a healthy reserve and disciplined spending",
            "Cash flow is clearly stable with ample room for new commitments",
            "Applicant presents a reassuringly strong and balanced budget",
            "Household finances seem well cushioned against routine surprises",
            "Client shows consistent prudence and comfortable liquidity",
            "Financial standing looks notably sound and well managed",
            "Applicant has a very steady track record and low overhead",
            "Budgetary position appears resilient with clear surplus capacity",
            "Client's financial habits reflect strong stability and foresight",
            "Overall picture is very positive with solid income coverage",
            "Applicant seems well prepared to handle repayments comfortably",
            "Financials look well structured with plenty of breathing room",
            "Client exhibits commendable discipline in managing obligations",
            "Cash reserves and ongoing earnings appear comfortably aligned",
            "Applicant's profile is reassuringly orderly and low-stress",
            "Spending appears well within means with healthy margins",
            "Client maintains a very clean and stable financial footing",
            "Overall financial health looks robust and well-cushioned",
            "Applicant shows clear capacity to service the facility smoothly",
            "Budget looks well balanced with strong ongoing stability",
            "Client's fiscal management appears thorough and dependable",
            "Financial position is notably comfortable and well-supported"
        ],
        2: [
            "Finances look generally steady with reasonable monthly headroom",
            "Applicant appears to manage regular bills without much difficulty",
            "Overall profile is mildly positive with adequate cash flow",
            "Client keeps a fairly balanced budget and steady routine",
            "Cash flow seems stable enough to support the requested plan",
            "Applicant shows decent organization in handling day-to-day costs",
            "Household budget appears reasonably sound and manageable",
            "Client maintains a satisfactory cushion for routine expenses",
            "Financial standing looks fairly solid with modest reserves",
            "Applicant has a generally reliable approach to monthly obligations",
            "Budgetary position seems comfortable for current commitments",
            "Client's financial habits appear mostly disciplined and steady",
            "Overall picture is encouraging with manageable outgoings",
            "Applicant seems capable of fitting payments into their budget",
            "Financials look reasonably orderly without obvious pressure",
            "Client exhibits steady management of existing responsibilities",
            "Earnings and expenses appear fairly well matched",
            "Applicant's profile is generally reassuring and stable",
            "Spending seems mostly controlled with some spare capacity",
            "Client maintains a decent footing for this stage of life",
            "Overall financial health looks satisfactory and balanced",
            "Applicant shows adequate capacity to take on the facility",
            "Budget looks fairly well kept with steady income support",
            "Client's fiscal routine appears reasonably dependable",
            "Financial position is mildly favorable and well-paced"
        ],
        3: [
            "Applicant was polite and straightforward during the intake interview",
            "Client answered all standard application questions clearly",
            "Discussion focused mainly on the timeline for processing",
            "Applicant seemed focused on understanding the repayment schedule",
            "Client inquired about standard terms and early settlement rules",
            "Conversation was brief and centered on routine application steps",
            "Applicant expressed interest in knowing the expected approval window",
            "Client appeared calm and matter-of-fact throughout the meeting",
            "Interview proceeded smoothly with no unusual remarks noted",
            "Applicant asked about how monthly installments are typically billed",
            "Client seemed primarily concerned with getting the paperwork right",
            "Discussion covered basic product features and standard fees",
            "Applicant was attentive while reviewing the general terms",
            "Client mentioned they had been comparing a few local lenders",
            "Meeting was routine and focused on verifying personal details",
            "Applicant asked whether automatic debit is required for payments",
            "Client appeared neutral and businesslike during the consultation",
            "Conversation stayed strictly on procedural requirements",
            "Applicant wanted clarity on the disbursement timeline",
            "Client seemed comfortable with the standard documentation checklist",
            "Interview was concise with all basic disclosures acknowledged",
            "Applicant asked about options for changing the due date later",
            "Client was cooperative and provided straightforward responses",
            "Discussion remained focused on general loan administration",
            "Applicant noted they prefer clear, predictable monthly billing"
        ],
        4: [
            "Applicant mentioned some recent tightness in their monthly budget",
            "Client noted that rising living costs have trimmed their spare cash",
            "Finances appear slightly stretched by existing commitments",
            "Applicant expressed mild concern about juggling upcoming bills",
            "Cash flow looks somewhat tight with limited room for error",
            "Client indicated that recent expenses have reduced their savings",
            "Budgetary margins appear a bit thin at the moment",
            "Applicant seemed slightly anxious about keeping up with outgoings",
            "Financials show minor strain from competing household priorities",
            "Client mentioned needing careful planning to cover monthly dues",
            "Overall position looks somewhat pressured by recent outlays",
            "Applicant noted a few tight months recently but expects improvement",
            "Spending appears close to current earnings with little cushion",
            "Client expressed a desire to keep installments as low as possible",
            "Cash reserves seem modest relative to upcoming obligations",
            "Applicant acknowledged that their budget has been stretched lately",
            "Financials reflect some mild pressure on monthly liquidity",
            "Client seemed cautious about taking on additional fixed costs",
            "Budget looks somewhat constrained by ongoing family expenses",
            "Applicant noted that timing of paychecks has been a minor challenge",
            "Overall picture shows slight strain without major disruption",
            "Client mentioned cutting back on discretionary items recently",
            "Financial footing appears a bit uneven over the past quarter",
            "Applicant expressed mild worry about unexpected household costs",
            "Cash flow seems somewhat tight, warranting careful budgeting"
        ],
        5: [
            "Applicant appeared noticeably stressed about mounting monthly bills",
            "Client reported significant difficulty keeping up with current dues",
            "Finances look heavily strained with very little liquidity remaining",
            "Applicant described ongoing cash flow shortfalls in recent months",
            "Budget appears severely stretched by competing financial pressures",
            "Client expressed clear distress over accumulating household obligations",
            "Cash reserves seem exhausted with pressing bills still pending",
            "Applicant noted they have been falling behind on routine payments",
            "Financials reflect acute pressure and minimal breathing room",
            "Client seemed overwhelmed by the weight of existing commitments",
            "Overall position looks fragile with urgent need for relief",
            "Applicant mentioned struggling to make ends meet each month",
            "Spending and obligations appear to be outpacing current income",
            "Client reported frequent overdrafts or tight end-of-month balances",
            "Budgetary strain is evident with little capacity for surprises",
            "Applicant acknowledged serious challenges in managing current debts",
            "Financials show clear signs of distress and tight liquidity",
            "Client seemed deeply worried about meeting upcoming due dates",
            "Cash flow appears critically tight with heavy ongoing demands",
            "Applicant described a tough stretch of unforeseen financial hits",
            "Overall picture indicates substantial strain on household finances",
            "Client mentioned having to defer other bills to stay afloat",
            "Financial footing looks precarious with very thin margins",
            "Applicant expressed urgent concern about stabilizing their budget",
            "Cash position appears severely pressured at the time of review"
        ]
    }

    neutral_details = [
        "All required identification documents were submitted at the branch.",
        "Client prefers to be contacted via email for status updates.",
        "Income verification slips were attached to the file.",
        "Applicant visited the downtown branch in person this morning.",
        "Bank statements for the past three months were provided.",
        "Client requested SMS notifications once the review is complete.",
        "Proof of address was verified against utility records.",
        "Applicant prefers phone calls in the late afternoon.",
        "Signed consent forms were uploaded through the portal.",
        "Client brought original ID and tax documents for inspection.",
        "Employment letter was included with the application packet.",
        "Applicant asked for digital copies of the final agreement.",
        "Follow-up contact is scheduled for early next week.",
        "Client confirmed their residential address has not changed.",
        "Supporting paperwork was complete at the time of submission.",
        "Applicant prefers written correspondence over phone calls.",
        "Branch officer verified the signature on the application form.",
        "Client submitted additional reference details upon request.",
        "Application was logged during a scheduled morning appointment.",
        "Applicant confirmed their primary phone number for alerts.",
        "Standard KYC checks were initiated during the visit.",
        "Client requested that statements be sent electronically.",
        "All pages of the application were initialed and dated.",
        "Applicant noted they will be reachable by mobile all week.",
        "File was routed to underwriting with complete attachments."
    ]

    remarks = []
    used_remarks = {}
    for i in range(n):
        p_list = purpose_phrases[loan_purpose[i]]
        t_list = tier_phrases[tone_tier[i]]
        for attempt in range(50):
            p_phrase = rng.choice(p_list)
            t_phrase = rng.choice(t_list)
            include_detail = rng.random() < 0.50
            d_phrase = rng.choice(neutral_details) if include_detail else ""

            order = rng.integers(0, 3)
            if include_detail:
                if order == 0:
                    text = f"{p_phrase}. {t_phrase}. {d_phrase}"
                elif order == 1:
                    text = f"{t_phrase}. {p_phrase}. {d_phrase}"
                else:
                    text = f"{d_phrase} {p_phrase}. {t_phrase}."
            else:
                if order == 0:
                    text = f"{p_phrase}. {t_phrase}."
                else:
                    text = f"{t_phrase}. {p_phrase}."

            words = text.split()
            if 12 <= len(words) <= 30 and used_remarks.get(text, 0) < 3:
                used_remarks[text] = used_remarks.get(text, 0) + 1
                remarks.append(text)
                break
        else:
            # Fallback if 50 attempts didn't satisfy
            used_remarks[text] = used_remarks.get(text, 0) + 1
            remarks.append(text)

    # Build DataFrames
    df_main = pd.DataFrame({
        "customer_id": customer_id,
        "age": age,
        "annual_income": annual_income,
        "credit_score": credit_score,
        "loan_amount": loan_amount,
        "debt_to_income_ratio": debt_to_income_ratio,
        "late_payments_12m": late_payments_12m,
        "account_tenure_months": account_tenure_months,
        "employment_status": employment_status,
        "loan_default": loan_default,
        "customer_remark": remarks
    })

    df_hidden = pd.DataFrame({
        "customer_id": customer_id,
        "loan_purpose": loan_purpose,
        "tone_tier": tone_tier,
        "stress": np.round(stress, 6)
    })

    df_main.to_csv("financial_loan_dataset_v2.csv", index=False)
    df_hidden.to_csv("hidden_ground_truth.csv", index=False)

    # STEP 5 - CHECKS
    print("=== VALIDATION CHECKS ===")
    print(f"1. Rows: {len(df_main)}, Missing values: {df_main.isna().sum().sum()}, Duplicate IDs: {df_main['customer_id'].duplicated().sum()}")
    default_rate = df_main["loan_default"].mean()
    print(f"2. Default rate: {default_rate:.4f} (target 0.18 - 0.22)")
    vc_remarks = df_main["customer_remark"].value_counts()
    print(f"3. Unique remarks: {len(vc_remarks)} (>= 1100), Max remark count: {vc_remarks.max()} (<= 3)")

    merged = df_main.merge(df_hidden, on="customer_id")
    tier_rates = merged.groupby("tone_tier")["loan_default"].mean()
    print("4. Default rate by tone_tier:")
    for t, r in tier_rates.items():
        print(f"   Tier {t}: {r:.4f}")

    tenure_ok = (df_main["account_tenure_months"] <= (df_main["age"] - 18) * 12).all()
    print(f"5. account_tenure_months <= (age - 18) * 12 for all rows: {tenure_ok}")

    max_freq_inc = df_main["annual_income"].value_counts().max() / n
    max_freq_loan = df_main["loan_amount"].value_counts().max() / n
    max_freq_dti = df_main["debt_to_income_ratio"].value_counts().max() / n
    print(f"6. Max single value freq - income: {max_freq_inc:.4f}, loan: {max_freq_loan:.4f}, dti: {max_freq_dti:.4f} (all <= 0.01)")

    # 7. Logistic regression on numeric and employment columns only, 5-fold ROC-AUC
    num_cols = ["age", "annual_income", "credit_score", "loan_amount", "debt_to_income_ratio", "late_payments_12m", "account_tenure_months"]
    cat_cols = ["employment_status"]
    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols)
    ])
    pipe_num = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_num = cross_val_score(pipe_num, df_main[num_cols + cat_cols], df_main["loan_default"], cv=cv, scoring="roc_auc").mean()
    print(f"7. Numeric + Employment 5-fold ROC-AUC: {auc_num:.4f} (target 0.75 - 0.90)")

    # 8. TF-IDF of customer_remark + logistic regression, 5-fold ROC-AUC
    pipe_txt = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000))
    ])
    auc_txt = cross_val_score(pipe_txt, df_main["customer_remark"], df_main["loan_default"], cv=cv, scoring="roc_auc").mean()
    print(f"8. TF-IDF Remark 5-fold ROC-AUC: {auc_txt:.4f} (target 0.60 - 0.80)")

if __name__ == "__main__":
    main()


<class 'ModuleNotFoundError'>: No module named 'pandas'